In [18]:
import sys
print(sys.executable)

c:\Users\KULPREET SEMBHI\OneDrive\Desktop\pw projects\pinecone_env\Scripts\python.exe


In [19]:
import os
from pathlib import Path

from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

In [20]:
load_dotenv(override=True)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("Pinecone API key not found. Check your .env file.")

print("API key loaded successfully.")

API key loaded successfully.


In [21]:
pc = Pinecone(api_key=PINECONE_API_KEY)

print(pc.list_indexes())

[{
    "name": "volvo2",
    "metric": "cosine",
    "host": "volvo2-ihxd5gb.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}]


In [22]:
index_name = "volvo2"

index_info = pc.describe_index(name=index_name)

index = pc.Index(host=index_info.host)

stats = index.describe_index_stats()

print(stats)

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 1964}},
 'total_vector_count': 1964,
 'vector_type': 'dense'}


In [23]:
db_pinecone = PineconeVectorStore(
    index=index,
    embedding=embedding_model
)

print("Pinecone vector store initialized successfully!")

Pinecone vector store initialized successfully!


In [24]:
query = "What are the safety features of the Volvo XC90?"

results = db_pinecone.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(doc.page_content)
    print("Metadata:", doc.metadata)
    


Result 1:
(CL) for the specific car. The information on substances in the candidate list (CL) is based on the data obtained from our suppli-
ers and our own product data.
General Safe Use Information for Articles
Every car from Volvo Cars is provided with a user manual, which includes safe use information for owners, drivers and users of
the car. Volvo Cars information on the repair and servicing of cars and genuine parts also includes safe use information for ser-
vice personnel.
Where present in parts of this car, the Candidate List substances shown in the relevant "Candidate List Substances Table" for
the specific car are incorporated in such a way that potential exposure to customers as well as risks for people or the environ-
ment can be minimised as long as the car and its parts are used as intended, and any repairs, servicing and maintenance are
carried out following technical instructions for those activities, and industry standard good practices.
Metadata: {'creationdate': '2

In [25]:
query = "What safety systems and driver assistance features are available in the Volvo XC90?"

results = db_pinecone.similarity_search(
    query,
    k=5
)

for i, doc in enumerate(results, 1):
    print(f"\n{'='*60}")
    print(f"RESULT {i}")
    print(f"Page: {doc.metadata.get('page_label')}")
    print(f"Source: {doc.metadata.get('source')}")
    print(doc.page_content)


RESULT 1
Page: 15
Source: 2027-volvo-xc90.pdf
Driving and using driver support functions
You are responsible for adapting your driving to the current conditions, even when using driver support functions. This includes
adapting your distance to other vehicles and speed as well as being ready to react to traffic and road hazards. The car's safety
interventions and warnings rely on accurate detection and identification of surrounding traffic and road conditions. The detec-
tion systems cannot handle all driving, traffic, weather and road conditions.
Knowing the car's capabilities and limitations
Before driving, all drivers are recommended to familiarise themselves with the car and any functions and features they might
use. The driver has a responsibility to ensure they have sufficient knowledge of the car to use it safely.
If you are uncertain about any of the car's functions or have questions about its intended use, consult the manual. If you can't
find the information you need, contact

In [26]:
from langchain_ollama import ChatOllama

In [27]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

response = llm.invoke("Explain what a RAG application is in two sentences.")

print(response.content)

I couldn't find any information on a "RAG application." It's possible that it's a specific term used in a particular context or industry, but without more information, I couldn't provide a definition. Can you please provide more context or clarify what a RAG application refers to?


In [28]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant that answers questions
about the Volvo XC90 2027 user manual.

Answer the question using ONLY the provided context.

If the answer is not available in the context,
clearly say that the information is not available
in the provided document.

Context:
{context}

Question:
{question}

Answer:
""")

In [29]:
def ask_question(question):

    docs = db_pinecone.similarity_search(
        question,
        k=5
    )

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    chain = prompt | llm

    response = chain.invoke({
        "context": context,
        "question": question
    })

    print("Answer:\n")
    print(response.content)

    print("\nSources:\n")

    for doc in docs:
        print(
            f"Page: {doc.metadata.get('page_label')} | "
            f"Source: {doc.metadata.get('source')}"
        )

In [30]:
ask_question(
    "What safety features are available in the Volvo XC90?"
)

Answer:

The provided context does not mention specific safety features available in the Volvo XC90. It does provide general information on safe use, warnings, and guidelines for the car, but it does not specifically mention the safety features of the Volvo XC90.

Sources:

Page: 428 | Source: 2027-volvo-xc90.pdf
Page: 428 | Source: 2027-volvo-xc90.pdf
Page: 17 | Source: 2027-volvo-xc90.pdf
Page: 17 | Source: 2027-volvo-xc90.pdf
Page: 15 | Source: 2027-volvo-xc90.pdf


In [31]:
query = "What safety features are available in the Volvo XC90?"

results_with_scores = db_pinecone.similarity_search_with_score(
    query,
    k=10
)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"\n{'='*60}")
    print(f"RESULT {i}")
    print(f"Similarity score: {score}")
    print(f"Page: {doc.metadata.get('page_label')}")
    print(f"Content:\n{doc.page_content[:700]}")


RESULT 1
Similarity score: 0.644199908
Page: 428
Content:
(CL) for the specific car. The information on substances in the candidate list (CL) is based on the data obtained from our suppli-
ers and our own product data.
General Safe Use Information for Articles
Every car from Volvo Cars is provided with a user manual, which includes safe use information for owners, drivers and users of
the car. Volvo Cars information on the repair and servicing of cars and genuine parts also includes safe use information for ser-
vice personnel.
Where present in parts of this car, the Candidate List substances shown in the relevant "Candidate List Substances Table" for
the specific car are incorporated in such a way that potential exposure to customers as well as ri

RESULT 2
Similarity score: 0.644061565
Page: 428
Content:
(CL) for the specific car. The information on substances in the candidate list (CL) is based on the data obtained from our suppli-
ers and our own product data.
General Safe Use Inf

In [32]:
query = """
Explain the Volvo XC90's airbags, seat belts,
collision avoidance system, driver assistance,
blind spot information system and lane keeping assistance.
"""

results = db_pinecone.similarity_search(query, k=10)

for i, doc in enumerate(results, 1):
    print(f"\nRESULT {i}")
    print("Page:", doc.metadata.get("page_label"))
    print(doc.page_content[:700])


RESULT 1
Page: 14
roadside assistance. Volvo Assistance is available 24 hours a day, 7 days a week.
Press the assist button 
  in the car's ceiling or use the mobile app for the car to contact Volvo Assistance.
1.3. Driver responsibility
As a driver, you are responsible for doing everything possible to ensure your own safety as well as that of
your passengers and other road users.
Your knowledge, decisions and actions determine how safely you drive. Your car has features that, in certain situations, can
compensate for mistakes and lapses in judgement. However, they do not change where the responsibility lies. They are a sup-
plement to good driver practices, which you are responsible for as a driver.
You have

RESULT 2
Page: 14
roadside assistance. Volvo Assistance is available 24 hours a day, 7 days a week.
Press the assist button 
  in the car's ceiling or use the mobile app for the car to contact Volvo Assistance.
1.3. Driver responsibility
As a driver, you are responsible for doin

In [33]:
from pypdf import PdfReader

pdf_path = "2027-volvo-xc90.pdf"

reader = PdfReader(pdf_path)

keywords = [
    "airbag",
    "collision avoidance",
    "blind spot",
    "lane keeping",
    "driver support",
    "safety",
    "seat belt",
    "parking assistance"
]

for page_num, page in enumerate(reader.pages):
    text = page.extract_text() or ""

    matched = [
        word for word in keywords
        if word.lower() in text.lower()
    ]

    if matched:
        print(f"\nPage {page_num + 1}: {matched}")
        print(text[:500])


Page 1: ['safety']
XC90 2027 (26w17) User Manual
Version 2026-04-27
Disclaimer
Due to the dynamic nature of our software-based product, the content of this PDF represents the most up-to-date version of
the user manual as of the time of printing. As we continuously update and improve our product, certain content may not reflect
the most up-to-date information in a future instance. Therefore, we strongly recommend utilising the digital user manual app in
your car's centre display for the most accurate and up-to-date

Page 4: ['airbag', 'safety']
4.5.3 Disabling interior auto lights
4.5.4 Activating all interior lights
5. Safety
5.1 Collision response
5.2 Proper seating
5.3 Seatbelts
5.3.1 Fastening and adjusting seatbelt
5.3.2 Seatbelt reminder
5.4 Airbags
5.4.1 Airbag deployment
5.4.2 Front airbags
5.4.2.1 Disabling and enabling front passenger airbag
5.4.3 Side airbags
5.4.4 Inflatable curtains
5.4.5 Airbag maintenance and servicing
5.4.6 Airbag labels
5.5 Child safety
5.5.1 Child res

In [34]:
query = "Volvo XC90 airbags, seatbelts, collision protection, lane keeping aid, blind spot information and collision warnings"

results = db_pinecone.similarity_search(query, k=10)

for i, doc in enumerate(results, 1):
    print(f"\n{'='*60}")
    print(f"RESULT {i}")
    print("PDF Page:", doc.metadata.get("page_label"))
    print("Content:", doc.page_content[:1000])


RESULT 1
PDF Page: 130
Content: Reading the manual allows you to recognise and avoid practices known to increase the risk of injury.
To reduce the risk of airbag-related injuries in a collision:
Follow the manual’s instructions for proper seating and use of seatbelts.
Learn the placement of all airbags and how they affect the use of your car.
Properly stow loose objects when driving and do not place or mount any objects around the airbag deployment
areas.
Do not make any modifications to the interior or electrical systems of your car that are not approved by Volvo.
The content of this manual represents the status of the user manual at the time of printing and may not be completely valid in future
instances. For more information, refer to the first page for the complete disclaimer note.
130 / 443

RESULT 2
PDF Page: 47
Content: Collision risk warning There is a risk of collision with another vehicle, a pedestrian, a cyclist or a large animal.
High engine temperature The engine is too h

In [35]:
retriever = db_pinecone.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)

docs = retriever.invoke(
    "What are the airbags, collision prevention and driver assistance features of the Volvo XC90?"
)

for i, doc in enumerate(docs, 1):
    print(f"\nRESULT {i}")
    print("Page:", doc.metadata.get("page_label"))
    print(doc.page_content[:1000])


RESULT 1
Page: 130
Reading the manual allows you to recognise and avoid practices known to increase the risk of injury.
To reduce the risk of airbag-related injuries in a collision:
Follow the manual’s instructions for proper seating and use of seatbelts.
Learn the placement of all airbags and how they affect the use of your car.
Properly stow loose objects when driving and do not place or mount any objects around the airbag deployment
areas.
Do not make any modifications to the interior or electrical systems of your car that are not approved by Volvo.
The content of this manual represents the status of the user manual at the time of printing and may not be completely valid in future
instances. For more information, refer to the first page for the complete disclaimer note.
130 / 443

RESULT 2
Page: 4
4.5.3 Disabling interior auto lights
4.5.4 Activating all interior lights
5. Safety
5.1 Collision response
5.2 Proper seating
5.3 Seatbelts
5.3.1 Fastening and adjusting seatbelt
5.3.2 Se

In [36]:
print("Embedding model:", embedding_model)
print("Vector dimension:", len(embedding_model.embed_query("test query")))

Embedding model: model_name='all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False
Vector dimension: 384


In [37]:
print("Total PDF pages:", len(pages))
print("Total chunks:", len(chunks))

print("\nFirst chunk metadata:")
print(chunks[0].metadata)

print("\nFirst chunk content:")
print(chunks[0].page_content[:500])

print("\nPinecone index statistics:")
print(index.describe_index_stats())

Total PDF pages: 443
Total chunks: 1060

First chunk metadata:
{'producer': 'iLovePDF', 'creator': 'Chromium', 'creationdate': '2026-04-27T01:03:39+00:00', 'title': 'XC90 2027 (26w17) User Manual', 'moddate': '2026-06-17T15:57:31+00:00', 'source': '2027-volvo-xc90.pdf', 'total_pages': 443, 'page': 0, 'page_label': '1', 'text': "XC90 2027 (26w17) User Manual\nVersion 2026-04-27\nDisclaimer\nDue to the dynamic nature of our software-based product, the content of this PDF represents the most up-to-date version of\nthe user manual as of the time of printing. As we continuously update and improve our product, certain content may not reflect\nthe most up-to-date information in a future instance. Therefore, we strongly recommend utilising the digital user manual app in\nyour car's centre display for the most accurate and up-to-date information. You can also access information in the Volvo Cars\nmobile app.\nPlease note that if you choose to print the manual, we cannot guarantee the validity o

In [38]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone

load_dotenv(override=True)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("Pinecone API key not found in .env")

pc = Pinecone(api_key=PINECONE_API_KEY)

print("Pinecone connection successful!")
print(pc.list_indexes())

Pinecone connection successful!
[{
    "name": "volvo2",
    "metric": "cosine",
    "host": "volvo2-ihxd5gb.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}]


In [39]:
index_info = pc.describe_index(name="volvo2")
index = pc.Index(host=index_info.host)

print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 1964}},
 'total_vector_count': 1964,
 'vector_type': 'dense'}


In [40]:
print("Total PDF pages:", len(pages))
print("Total chunks:", len(chunks))

print("\nFirst chunk metadata:")
print(chunks[0].metadata)

print("\nFirst chunk content:")
print(chunks[0].page_content[:500])

print("\nPinecone index statistics:")
print(index.describe_index_stats())

Total PDF pages: 443
Total chunks: 1060

First chunk metadata:
{'producer': 'iLovePDF', 'creator': 'Chromium', 'creationdate': '2026-04-27T01:03:39+00:00', 'title': 'XC90 2027 (26w17) User Manual', 'moddate': '2026-06-17T15:57:31+00:00', 'source': '2027-volvo-xc90.pdf', 'total_pages': 443, 'page': 0, 'page_label': '1', 'text': "XC90 2027 (26w17) User Manual\nVersion 2026-04-27\nDisclaimer\nDue to the dynamic nature of our software-based product, the content of this PDF represents the most up-to-date version of\nthe user manual as of the time of printing. As we continuously update and improve our product, certain content may not reflect\nthe most up-to-date information in a future instance. Therefore, we strongly recommend utilising the digital user manual app in\nyour car's centre display for the most accurate and up-to-date information. You can also access information in the Volvo Cars\nmobile app.\nPlease note that if you choose to print the manual, we cannot guarantee the validity o

In [41]:
# Create a separate namespace for the current PDF

namespace = "volvo_xc90_manual_v1"

db_pinecone = PineconeVectorStore(
    index=index,
    embedding=embedding_model,
    namespace=namespace
)

print("Vector store initialized successfully!")
print("Namespace:", namespace)

Vector store initialized successfully!
Namespace: volvo_xc90_manual_v1


In [42]:
stats = index.describe_index_stats()

namespace_stats = stats.namespaces.get(namespace)

if namespace_stats and namespace_stats.vector_count > 0:
    print("Documents already uploaded:", namespace_stats.vector_count)
    print("Skipping upload to avoid duplicate vectors.")

else:
    print("Uploading PDF chunks to Pinecone...")

    db_pinecone.add_documents(chunks)

    print("Upload completed successfully!")

Uploading PDF chunks to Pinecone...
Upload completed successfully!


In [43]:
stats = index.describe_index_stats()

print("Total vectors:", stats.total_vector_count)
print("Namespace details:", stats.namespaces)

Total vectors: 3024
Namespace details: {'volvo_xc90_manual_v1': {'vector_count': 1060}, '': {'vector_count': 1964}}


In [44]:
query = "What airbags are fitted in the Volvo XC90 and how do they protect occupants?"

results = db_pinecone.similarity_search(query, k=5)

for i, doc in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Page:", doc.metadata.get("page_label", "Not available"))
    print("Content:")
    print(doc.page_content[:800])
    print("-" * 80)


Result 1
Page: 135
Content:
5.4.3. Side airbags
The side airbags are designed to deploy in the event of a side-on collision.
The side airbags can help protect the driver and front passenger, if they are properly seated. The side airbags will typically only
deploy on the collision side of each front seat.
Side collision airbags for the front occupants.
The side airbags are placed on the side of each front seat that is closest to the door. The airbags are packed into the seat's back
frame.
Both front seats have markings with the text AIRBAG where the airbags are placed.
 Warning
Do not block the side airbags
Do not place any objects on the side of the front seats. Objects between the seats and the door panel can interfere
with the side airbags.
Do not use seat covers that have not been approved by Volvo.
Blocking airba
--------------------------------------------------------------------------------

Result 2
Page: 131
Content:
After airbag deployment
After a collision in which the airba

In [45]:
retriever = db_pinecone.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 15,
        "lambda_mult": 0.5
    }
)

query = "What airbags are fitted in the Volvo XC90 and how do they protect occupants?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"\nResult {i}")
    print("Page:", doc.metadata.get("page_label", "Not available"))
    print("Content:")
    print(doc.page_content[:800])
    print("-" * 80)


Result 1
Page: 135
Content:
5.4.3. Side airbags
The side airbags are designed to deploy in the event of a side-on collision.
The side airbags can help protect the driver and front passenger, if they are properly seated. The side airbags will typically only
deploy on the collision side of each front seat.
Side collision airbags for the front occupants.
The side airbags are placed on the side of each front seat that is closest to the door. The airbags are packed into the seat's back
frame.
Both front seats have markings with the text AIRBAG where the airbags are placed.
 Warning
Do not block the side airbags
Do not place any objects on the side of the front seats. Objects between the seats and the door panel can interfere
with the side airbags.
Do not use seat covers that have not been approved by Volvo.
Blocking airba
--------------------------------------------------------------------------------

Result 2
Page: 129
Content:
experienced by occupants.
The image shows a selection of ava

In [46]:
retriever = db_pinecone.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 15,
        "lambda_mult": 0.8
    }
)

query = "What airbags are fitted in the Volvo XC90 and how do they protect occupants?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"\nResult {i}")
    print("Page:", doc.metadata.get("page_label", "Not available"))
    print(doc.page_content[:800])
    print("-" * 80)


Result 1
Page: 135
5.4.3. Side airbags
The side airbags are designed to deploy in the event of a side-on collision.
The side airbags can help protect the driver and front passenger, if they are properly seated. The side airbags will typically only
deploy on the collision side of each front seat.
Side collision airbags for the front occupants.
The side airbags are placed on the side of each front seat that is closest to the door. The airbags are packed into the seat's back
frame.
Both front seats have markings with the text AIRBAG where the airbags are placed.
 Warning
Do not block the side airbags
Do not place any objects on the side of the front seats. Objects between the seats and the door panel can interfere
with the side airbags.
Do not use seat covers that have not been approved by Volvo.
Blocking airba
--------------------------------------------------------------------------------

Result 2
Page: 131
After airbag deployment
After a collision in which the airbags have deployed, 

In [47]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

response = llm.invoke("Explain what an airbag does in one simple sentence.")

print(response.content)


An airbag is a safety device that inflates rapidly to cushion and protect the occupants of a vehicle in the event of a collision, reducing the impact of the crash on the body.


In [52]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant specialised in the Volvo XC90 2027 user manual.

Answer the question using ONLY the provided context.

Instructions:
1. Read all the provided context carefully before answering.
2. Include all relevant information available in the context.
3. Do not omit important categories, types, features or specifications.
4. When the question asks for types or categories, list every type explicitly mentioned in the context.
5. Explain the answer clearly using numbered points where appropriate.
6. Never invent information or use outside knowledge.
7. If information is missing, clearly mention that it is not available in the provided manual.
8. Mention the relevant PDF page numbers.

Context:
{context}

Question:
{question}

Answer:
""")

print("Improved RAG prompt created successfully!")

Improved RAG prompt created successfully!


In [53]:
def ask_question(question):

    # Step 1: Retrieve relevant documents from Pinecone
    docs = retriever.invoke(question)

    # Step 2: Prepare context with PDF page numbers
    context_parts = []

    for doc in docs:
        page = doc.metadata.get("page_label", "Unknown")
        content = doc.page_content

        context_parts.append(
            f"[PDF Page {page}]\n{content}"
        )

    context = "\n\n".join(context_parts)

    # Step 3: Generate answer using Ollama
    rag_chain = prompt | llm

    response = rag_chain.invoke({
        "context": context,
        "question": question
    })

    # Step 4: Collect source pages
    source_pages = sorted(set(
        doc.metadata.get("page_label", "Unknown")
        for doc in docs
    ))

    # Step 5: Display answer and sources
    print("Question:", question)
    print("\nAnswer:")
    print(response.content)

    print("\nSource PDF pages:", ", ".join(map(str, source_pages)))

    return response.content


In [54]:
question = "What types of airbags are available in the Volvo XC90 2027?"

answer = ask_question(question)

Question: What types of airbags are available in the Volvo XC90 2027?

Answer:
Based on the provided context, the following types of airbags are available in the Volvo XC90 2027:

1. Front airbags
2. Side airbags

Specifically, the context mentions:

* Front airbags (5.4.2) - deployed in certain frontal collisions, can help protect the driver and front passenger from severe injury if properly seated.
* Side airbags (5.4.3) - designed to deploy in the event of a side-on collision, can help protect the driver and front passenger if properly seated.

Note that the context does not provide information on other types of airbags, such as knee airbags or side curtain airbags, which are commonly found in modern vehicles.

Source PDF pages: 120, 130, 131, 135, 138


In [55]:
question = "What types of airbags are available in the Volvo XC90 2027?"

docs = retriever.invoke(question)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Document {i} ---")
    print("Page:", doc.metadata.get("page_label"))
    print(doc.page_content)
    print("-" * 80)
    


--- Document 1 ---
Page: 135
5.4.3. Side airbags
The side airbags are designed to deploy in the event of a side-on collision.
The side airbags can help protect the driver and front passenger, if they are properly seated. The side airbags will typically only
deploy on the collision side of each front seat.
Side collision airbags for the front occupants.
The side airbags are placed on the side of each front seat that is closest to the door. The airbags are packed into the seat's back
frame.
Both front seats have markings with the text AIRBAG where the airbags are placed.
 Warning
Do not block the side airbags
Do not place any objects on the side of the front seats. Objects between the seats and the door panel can interfere
with the side airbags.
Do not use seat covers that have not been approved by Volvo.
Blocking airbags in general
Keep all airbag locations and expansion spaces free of obstructions. Obstructions can reduce airbag effectiveness and
cause serious injury.
----------------

In [56]:
query = "Airbag types: front airbags, side airbags, inflatable curtains, knee airbags and their locations"

results = db_pinecone.similarity_search(
    query,
    k=8
)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Page:", doc.metadata.get("page_label"))
    print(doc.page_content)
    print("-" * 80)


--- Result 1 ---
Page: 130
The airbags are designed to work with the rest of the car's safety features. Their effectiveness particularly relies on proper seat-
ing and seatbelt use. An airbag deployment is a sudden, forceful but controlled event that can significantly lower the risk of seri-
ous injury for correctly seated occupants.
Sensors throughout your car allow it to deploy different airbags based on information about the collision itself, as well as the
status of the car and its occupants.
Airbag types
Your car has the following airbag types:
Front airbags Frontal collision airbags for the front occupants.
Side airbags Seat-integrated side-on collision airbags for the front occupants.
Inflatable curtains Ceiling-mounted airbags for occupants seated by a window.
5.4.1. Airbag deployment
If an airbag has deployed, your car needs to be recovered and serviced.
When the car deploys an airbag, it inflates almost instantly with considerable force accompanied by a loud noise. After tha

In [57]:
retriever = db_pinecone.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)

print("Similarity retriever initialized successfully!")

Similarity retriever initialized successfully!


In [58]:
question = "What types of airbags are available in the Volvo XC90 2027? Explain their purpose."

answer = ask_question(question)

Question: What types of airbags are available in the Volvo XC90 2027? Explain their purpose.

Answer:
Based on the provided context, the following types of airbags are available in the Volvo XC90 2027:

1. Front airbags: Designed to deploy in certain frontal collisions, these airbags can help protect the driver and front passenger from severe injury if they are properly seated during a collision.
2. Side airbags: Designed to deploy in the event of a side-on collision, these airbags can help protect the driver and front passenger, if they are properly seated. They are placed on the side of each front seat that is closest to the door.
3. Inflatable curtains: Ceiling-mounted airbags that deploy in the event of a collision, designed to protect occupants seated by a window.

These airbags are designed to work together with the rest of the car's safety features to provide optimal protection for the occupants. According to the manual, it is essential to follow the instructions for proper seat

In [59]:
question = "What is the fuel price in Ludhiana today?"

answer = ask_question(question)

Question: What is the fuel price in Ludhiana today?

Answer:
I must point out that the provided context does not contain any information about fuel prices in Ludhiana or any other location. The manual only provides information about the car's specifications, fuel consumption, and emissions, as well as general maintenance and repair procedures.

Therefore, I must inform you that the answer to your question is: **Not available in the provided manual**.

There is no information about fuel prices or current prices in the manual. If you need to know the current fuel price in Ludhiana, I recommend checking a reliable source such as a news website, fuel price tracking website, or a local news outlet.

Source PDF pages: 195, 369, 408, 409, 8


In [16]:
# 1) Create a account in pinecone website
# 2) Create a .env file and add your api key in the following format
!pip install pinecone -q
import sys
import subprocess

# Remove conflicting or previously installed versions
subprocess.run([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "pinecone", "pinecone-client", "langchain-pinecone"
])

# Install compatible versions
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "pinecone==7.3.0",
    "langchain-pinecone==0.2.13"
])

0

In [17]:
import sys
import pinecone

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Pinecone version:", pinecone.__version__)

%pip show langchain-pinecone

Python executable: c:\Users\KULPREET SEMBHI\OneDrive\Desktop\pw projects\pinecone_env\Scripts\python.exe
Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Pinecone version: 7.3.0
Name: langchain-pinecone
Version: 0.2.13
Summary: An integration package connecting Pinecone and LangChain
Home-page: 
Author: 
Author-email: 
License: MIT
Location: c:\Users\KULPREET SEMBHI\OneDrive\Desktop\pw projects\pinecone_env\Lib\site-packages
Requires: httpx, langchain-core, langchain-openai, numpy, pinecone, pydantic, simsimd
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pinecone import Pinecone, ServerlessSpec

In [4]:
import os, sys
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain.prompts import ChatPromptTemplate
from langchain_ollama import OllamaEmbeddings,OllamaLLM
from langchain_huggingface import HuggingFaceEmbeddings
DOC_PATH = "2027-volvo-xc90.pdf"
# CHROMA_PATH = "your_db_name" 

# ----- Data Indexing Process -----

# load your pdf doc
loader = PyPDFLoader(DOC_PATH)
pages = loader.load()

# split the doc into smaller chunks i.e. chunk_size=500
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

# get Ollama Embedding model
# embeddings = OllamaEmbeddings(model="nomic-embed-text")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")



C:\Users\KULPREET SEMBHI\AppData\Local\Temp\ipykernel_24124\3066416228.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
from dotenv import load_dotenv

load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

In [6]:
pc = Pinecone(api_key=PINECONE_API_KEY)

In [7]:
index_name = "volvo2"

if index_name not in [i["name"] for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        spec = ServerlessSpec(
            cloud="aws",
            region="us-east-1"),
    
    )
index = pc.Index(index_name)


In [8]:
print(f"PINECONE_API_KEY: {PINECONE_API_KEY}")

PINECONE_API_KEY: pcsk_77T5rB_LCcSZoq9VdmciDp491JVVDAdrKmXffXCRgy2AoTMyGVb9ktqxique2thdgovdGB


In [9]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\KULPREET SEMBHI\OneDrive\Desktop\pw projects\pinecone_env\Scripts\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [13]:
!pip show langchain-pinecone

Name: langchain-pinecone
Version: 0.2.13
Summary: An integration package connecting Pinecone and LangChain
Home-page: 
Author: 
Author-email: 
License: MIT
Location: C:\Users\KULPREET SEMBHI\OneDrive\Desktop\pw projects\pinecone_env\Lib\site-packages
Requires: httpx, langchain-core, langchain-openai, numpy, pinecone, pydantic, simsimd
Required-by: 
